# Teacher cache + student training (steps 1, 2, 4, 5) — just Run all

**Drive layout** (copy from the laptop, same paths):
```
MyDrive/model_KD/
  .env                                    API keys (private: keep this Drive folder unshared)
  configs/  student/  teacher/            code (whole folders)
  Data/Processed/canary/                  items.parquet, kb.json, lexicon.json (regenerated if missing)
  Data/Processed/cache/                   optional: gate.json from the laptop (the gate is then skipped)
```

**Two passes, both Run all.** Everything is resumable; finished work is skipped.
1. **CPU runtime** (no GPU quota used): key check, teacher gate, teacher cache (API calls, about 1-2 h). GPU cells are skipped.
2. **T4 GPU runtime**: hardware timing, untrained-student gate, scaling curves (about 2-3 h for the 0.5B student).

If a cell stops on a rate limit or a disconnect, run all again later: nothing is lost (outputs live on Drive).

In [ ]:
import json, os, sys
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/model_KD')
sys.path.insert(0, os.getcwd())
!pip install -q peft accelerate duckdb
!pip uninstall -q -y torchao  # Colab ships torchao 0.10; current peft rejects < 0.16, and this project does not use it
import torch
GPU = torch.cuda.is_available()
RUN_1_5B = False   # set True for the 1.5B scaling curve too (about 3x the 0.5B time)
os.makedirs('runs', exist_ok=True)
assert os.path.exists('.env'), 'upload your .env to MyDrive/model_KD/'
print('GPU:', torch.cuda.get_device_name(0) if GPU else 'none -> API steps only, GPU steps skipped')

In [4]:
# Canary data (seeded, so regenerating gives the same items). Expected counts:
# kb adapter 1500 / eval 1500 / pool 3000, lang adapter 3000 / eval 500 / pool 3000
if not os.path.exists('Data/Processed/canary/items.parquet'):
    !python -m teacher.canary
import duckdb
print(duckdb.sql("SELECT capability, split, count(*) FROM 'Data/Processed/canary/items.parquet' GROUP BY ALL ORDER BY ALL").fetchall())

[('canary_kb', 'adapter', 1500), ('canary_kb', 'eval', 1500), ('canary_kb', 'pool', 3000), ('canary_lang', 'adapter', 3000), ('canary_lang', 'eval', 500), ('canary_lang', 'pool', 3000)]
[('canary_kb', 'adapter', 1500), ('canary_kb', 'eval', 1500), ('canary_kb', 'pool', 3000), ('canary_lang', 'adapter', 3000), ('canary_lang', 'eval', 500), ('canary_lang', 'pool', 3000)]


## Teacher (API): key check, step-2 gate, step-4 cache

In [5]:
!python -m teacher.cache check

/usr/bin/python3: No module named teacher.cache


In [6]:
# Gate: teacher must score >= 90% on both capabilities (held-out eval items). Resumes from gate.json.
!python -m teacher.cache gate

/usr/bin/python3: No module named teacher.cache


In [7]:
# Cache: 6,000 pool prompts x (4 full + 1 answer-only). Resumable; rerun if it stops on a rate limit.
meta_f = 'Data/Processed/cache/teacher_cache.json'
if not (os.path.exists(meta_f) and json.load(open(meta_f))['complete']):
    !python -m teacher.cache build
CACHE_OK = os.path.exists(meta_f) and json.load(open(meta_f))['complete']
print('cache complete:', CACHE_OK, json.load(open(meta_f)) if os.path.exists(meta_f) else '')

/usr/bin/python3: No module named teacher.cache
cache complete: False 


## Student (GPU): step 1 timing, untrained-student gate, step 5 scaling curve

In [11]:
from configs import load
S = load()['student']
MODELS = ['Qwen/Qwen2.5-0.5B-Instruct'] + (['Qwen/Qwen2.5-1.5B-Instruct'] if RUN_1_5B else [])
if GPU:
    for m in MODELS:
        !python -m student.train_lora --model {m} --data canary-adapter --time-steps 20 | tee -a runs/timing_colab.txt
    steps = sum(int(3 * n / 8) for n in S['sizes']) * len(S['seeds'])  # 3 epochs, batch 8
    for r in [json.loads(l) for l in open('runs/timing_colab.txt') if l.startswith('{')][-len(MODELS):]:
        print(f"{r['model']}: {r['s_per_step']} s/step on {r['gpu']} -> training {steps * r['s_per_step'] / 3600:.1f} h "
              f"for {len(S['sizes'])} sizes x {len(S['seeds'])} seeds, plus evaluation")
else:
    print('skipped: no GPU')

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 290/290 [00:00<00:00, 964.41it/s]
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/drive/MyDrive/model_KD/student/train_lora.py", line 116, in <module>
    main()
    ~~~~^^
  File "/content/drive/MyDrive/model_KD/student/train_lora.py", line 65, in main
    model = get_peft_model(model, LoraConfig(r=a.rank, lora_alpha=2 * a.rank, lora_dropout=0.05, task_type="CAUSAL_LM",
                                             target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                                             "gate_proj", "up_proj", "down_proj"]))
  File "/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py", line 198, in get_peft_model
    return MODEL_TYPE_TO_PEFT_MODEL_MAPPING[peft_config.task_type](
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

In [12]:
# Step 2 gate, student side: the untrained student must score < 5% on the canary
if GPU and not os.path.exists('runs/gate_untrained_student.txt'):
    for m in MODELS:
        !python -m teacher.eval_canary --kind local --model {m} --n {S['eval_n']} | tee -a runs/gate_untrained_student.txt
print(open('runs/gate_untrained_student.txt').read() if os.path.exists('runs/gate_untrained_student.txt') else 'skipped')

In [ ]:
# Step 5 scaling curve: sizes x seeds per model; finished cells are skipped
if GPU and CACHE_OK:
    for m in MODELS:
        !python -m student.scaling --model {m}
else:
    print('skipped:', 'no GPU' if not GPU else 'teacher cache not complete yet')

In [ ]:
# Results so far: mean canary accuracy per (model, n) over seeds
if os.path.exists('runs/scaling/results.jsonl'):
    print(duckdb.sql("""SELECT model, n, count(*) seeds, round(avg(accuracy.canary_kb), 3) kb, round(avg(accuracy.canary_lang), 3) lang
        FROM read_json_auto('runs/scaling/results.jsonl') GROUP BY ALL ORDER BY ALL"""))